# Classical Experiment 1 — Thermodynamics of Black-Body Radiation (graphical)
*Background for Weinberg §1.1 ("Black-body radiation") — the experiments that forced Planck's law.*

**What the experiments showed (and where classical physics broke)**

- **Kirchhoff (1859–60):** cavity radiation is *universal* — a single function $\rho(\nu,T)$ for all materials in equilibrium. Thermodynamics alone can never fix its form: this is the opening contradiction.
- **Stefan (1879) / Boltzmann (1884):** total emission $\propto T^4$. Thermodynamics + radiation pressure $p = u/3$ gives $u = a_{\rm B}T^4$ *exactly* — but nothing about the spectral shape.
- **Wien (1893):** displacement law $\lambda_{\max}T = b$, and the scaling form $\rho = \nu^3 f(\nu/T)$.
- **Wien's exponential law (1896)** fit the visible spectrum but failed in the far infrared (Rubens & Kurlbaum 1900).
- **Rayleigh (1900) / Jeans (1905):** equipartition gives $\rho = 8\pi k_{\rm B}T\nu^2/c^3$ — exact in the far IR but divergent at high $\nu$: the **ultraviolet catastrophe**, infinite total energy.
- **Planck (1900):** one formula, $\rho = \dfrac{8\pi h}{c^3}\dfrac{\nu^3}{e^{h\nu/k_{\rm B}T}-1}$, matching both limits and fixing $h$ and $k_{\rm B}$.

**Plan:** every phenomenon above drawn — the $T^4$ law, the marching peaks, the spectral forms on one log–log map with the IR / visible / UV bands marked, and the divergent vs convergent integrated energy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from weinberg_constants import h, hbar, kB, c, a_B, sigma

def rho_planck(nu, T):
    with np.errstate(over='ignore'):
        return 8*np.pi*h/c**3 * nu**3 / np.expm1(np.minimum(h*nu/(kB*T), 700.0))

def rho_rayleigh_jeans(nu, T):
    return 8*np.pi*kB*T*nu**2/c**3

def rho_wien_exp(nu, T):
    return 8*np.pi*h/c**3 * nu**3 * np.exp(-h*nu/(kB*T))

print(f"sigma = {sigma:.5e} erg cm^-2 s^-1 K^-4  (Stefan-Boltzmann)")
print(f"a_B   = {a_B:.5e} erg cm^-3 K^-4        (radiation density)")

## 1. Stefan–Boltzmann: $E/A = \sigma T^4$
Stefan (1879) guessed the power law from Tyndall's platinum data; Boltzmann (1884) derived it from thermodynamics (shown symbolically below). Lummer & Pringsheim (1897–1900) verified it over $T \approx 1000$–$1600$ K.

**Graphical test:** $\log(\sigma T^4)$ vs $\log T$ must be a line of slope exactly 4.

In [ ]:
T = np.linspace(300, 2000, 200)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.loglog(T, sigma*T**4, lw=2, label=r'$P/A = \sigma T^4$')
ax.loglog(T, sigma*(T/2)**4 * (T*2/T)**4 / (T/2)**4 * (T/2)**4, alpha=0)  # placeholder no-op
# slope-4 reference line anchored at 1000 K
T0 = 1000.0
ax.loglog(T, sigma*T0**4 * (T/T0)**4, 'k--', lw=1, label='slope exactly 4')
for Tx in (373, 1000, 1600):
    ax.plot([Tx], [sigma*Tx**4], 'o')
    ax.annotate(f'{Tx:.0f} K', (Tx, sigma*Tx**4), textcoords='offset points', xytext=(6,-14), fontsize=9)
ax.set_xlabel(r'$T$ (K)'); ax.set_ylabel(r'$P/A$ (erg cm$^{-2}$ s$^{-1}$)')
ax.set_title('Stefan-Boltzmann law: log-log slope = 4')
ax.legend(); plt.show()
print(f"slope check (numeric d ln P / d ln T) = {np.polyfit(np.log(T), np.log(sigma*T**4), 1)[0]:.4f}")

## 2. The shape cannot be fixed by thermodynamics — two spectra, both $T^4$
Wien's exponential law and Planck's law are *different* spectra, yet both have the form $\nu^3 f(\nu/T)$, so both integrate to $u = a T^4$ with *different* constants. Experiment must decide the shape; thermodynamics only fixes the power of $T$.

In [ ]:
from scipy.integrate import quad
for T_test in (300.0, 1000.0):
    u_planck, _ = quad(lambda nu: rho_planck(nu, T_test), 0, np.inf, epsrel=1e-12)
    u_wien,   _ = quad(lambda nu: rho_wien_exp(nu, T_test), 0, np.inf, epsrel=1e-12)
    print(f"T = {T_test:6.0f} K:  u_Planck = {u_planck:.4e}, u_Wien = {u_wien:.4e}  (a_B T^4 = {a_B*T_test**4:.4e})")
    print(f"            ratio u_Wien/u_Planck = {u_wien/u_planck:.4f}  (= 90/pi^4 = {90/np.pi**4:.4f})")

## 3. Wien displacement law: $\lambda_{\max}T = b$
Wien's adiabatic argument (1893): slow expansion cools radiation so that $\nu/T$ is invariant, forcing $\rho = \nu^3 f(\nu/T)$. A peak in wavelength then satisfies $\lambda_{\max}T = b$ with $b = hc/(4.9651\,k_{\rm B})$ — Weinberg's Eq. (1.1.6) note on $a_{\rm B}$ uses the same scaling.

**Graphical test:** Planck curves at several $T$; the peaks must march along the hyperbola $\lambda_{\max} = b/T$.

In [ ]:
# spectral density per unit wavelength: rho_lambda = rho_nu * |dnu/dlambda| = rho_nu * c/lambda^2
def rho_lambda(lam, T):
    nu = c/lam
    return rho_planck(nu, T) * c/lam**2

b = h*c/(4.9651*kB)   # Wien displacement constant, cm K
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
temps = [1000, 1500, 2000, 3000, 5800]
lam = np.logspace(-5, -2, 600)  # cm : 100 nm .. 100 um
colors = plt.cm.viridis(np.linspace(0, 1, len(temps)))
for Tcol, Tx in zip(colors, temps):
    axes[0].plot(lam*1e4, rho_lambda(lam, Tx), color=Tcol, lw=2, label=f'{Tx} K')
    lmax = b/Tx
    axes[0].plot([lmax*1e4], [rho_lambda(lmax, Tx)], 'o', color=Tcol, ms=6)
axes[0].axvspan(0.38, 0.75, color='yellow', alpha=0.25)
axes[0].text(0.55, 0.02, 'visible', transform=axes[0].get_xaxis_transform(), ha='center', fontsize=9)
axes[0].set_xscale('log')
axes[0].set_ylabel(r'$\rho(\lambda,T)$ (erg cm$^{-4}$)')
axes[0].set_title('Planck spectra: peaks march as 1/T')
axes[0].legend(fontsize=8)

Tgrid = np.linspace(800, 6000, 200)
axes[1].plot(1/Tgrid*1e3, b/Tgrid*1e4, lw=2, label=r'$\lambda_{\max}=b/T$')
for Tx in temps:
    axes[1].plot([1/Tx*1e3], [b/Tx*1e4], 'o')
axes[1].set_xlabel(r'$10^3/T$ (K$^{-1}$)'); axes[1].set_ylabel(r'$\lambda_{\max}$ (µm)')
axes[1].set_title(f'Wien displacement: b = {b*1e4:.4f} cm·K ×10⁻⁴')
axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Thermodynamics alone: $u = a_{\rm B}T^4$ (symbolic)
First law $dU = T\,dS - p\,dV$ with $U = u(T)V$, Maxwell radiation pressure $p = u/3$, and the integrability condition $\left(\frac{\partial S}{\partial V}\right)_T = \left(\frac{\partial p}{\partial T}\right)_V$:

$$S = \frac{1}{T}\left(u + \frac{p}{1}\right)V \;\Rightarrow\; \frac{u+p}{T} = \left(\frac{\partial p}{\partial T}\right)_V
\;\Rightarrow\; \frac{4}{3}\frac{u}{T} = \frac{1}{3}\frac{du}{dT}.$$

In [ ]:
import sympy as sp
u, T, V = sp.symbols('u T V', positive=True)
u = sp.Function('u')
p = u(T)/3
integrability = sp.Eq((u(T)+p)/T, sp.diff(p, T))          # (u+p)/T = dp/dT
sol = sp.dsolve(integrability, u(T))
print("integrability condition:", integrability)
print("solution:", sol)
print("i.e. u = a_B * T^4 with a_B an undetermined constant -> sigma = c*a_B/4:", sp.simplify(c*a_B/4 - sigma) == 0)

## 5. The contradiction, drawn: Wien vs Rayleigh–Jeans vs Planck
One log–log map. The far-IR data (Rubens & Kurlbaum, 1900) kill Wien's exponential law; the UV extrapolation of Rayleigh–Jeans diverges. Planck's curve threads both regimes. Bands: far-IR ($\nu \lesssim 3\times10^{12}$ Hz), visible ($4$–$8\times10^{14}$ Hz), UV ($\gtrsim 10^{15}$ Hz).

In [ ]:
nu = np.logspace(11.5, 16.5, 700)
T = 1000.0
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.loglog(nu, rho_planck(nu, T), lw=2.5, color='k', label='Planck (1900)')
ax.loglog(nu, rho_rayleigh_jeans(nu, T), '--', lw=1.5, label='Rayleigh-Jeans: equipartition')
ax.loglog(nu, rho_wien_exp(nu, T), '--', lw=1.5, label='Wien exponential (1896)')
# shaded frequency bands
for lo, hi, lab in [(10**11.5, 3e12, 'far IR\n(Rubens-Kurlbaum)'), (4e14, 8e14, 'visible'), (1e15, 10**16.5, 'UV catastrophe\n(RJ diverges)')]:
    ax.axvspan(lo, hi, alpha=0.12, color=('tab:blue' if hi<1e13 else ('yellow' if hi<1e14 else 'tab:red')))
    ax.text(np.sqrt(lo*hi), 1e-12, lab, ha='center', fontsize=8)
ax.set_xlabel(r'$\nu$ (Hz)'); ax.set_ylabel(r'$\rho(\nu,T)$ (erg cm$^{-3}$ Hz$^{-1}$)')
ax.set_title(f'Black-body spectrum at T = {T:.0f} K: two classical failures, one quantum curve')
ax.legend(loc='lower center', fontsize=9)
plt.show()

## 6. Integrated energy: the catastrophe vs convergence
Cumulative $u(<\nu) = \int_0^\nu \rho(\nu',T)\,d\nu'$. Rayleigh–Jeans grows without bound ($\propto \nu^3$) — an infinite energy density at any $T$. Planck's saturates to the Stefan–Boltzmann value $a_{\rm B}T^4$.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
nu_c = np.logspace(11, 16.5, 400)
for Tx, cc in [(300.0, 'tab:blue'), (1000.0, 'tab:orange')]:
    cum_p = np.array([np.trapezoid(rho_planck(nu_c[nu_c<=v], Tx), nu_c[nu_c<=v]) for v in nu_c])
    cum_rj = 8*np.pi*kB*Tx/c**3 * nu_c**3 / 3
    ax.loglog(nu_c, cum_p, color=cc, lw=2, label=f'Planck, T={Tx:.0f} K')
    ax.loglog(nu_c, cum_rj, '--', color=cc, lw=1.2, label=f'Rayleigh-Jeans, T={Tx:.0f} K')
    ax.axhline(a_B*Tx**4, color=cc, ls=':', lw=1)
ax.text(1e15, a_B*1000.0**4*1.3, r'$a_{\rm B}T^4$', fontsize=9)
ax.set_xlabel(r'cutoff frequency $\nu$ (Hz)'); ax.set_ylabel(r'$u(<\nu)$ (erg cm$^{-3}$)')
ax.set_title('Cumulative energy: RJ has no ceiling')
ax.legend(fontsize=8); plt.show()

## My summary
- Thermodynamics is powerful but blind: Kirchhoff universality, the $T^4$ law, and $\lambda_{\max}T = b$ all follow without knowing $\rho(\nu,T)$ — but they cannot pick between Wien and Planck.
- Every classical candidate failed *somewhere*: Wien's law in the far IR, Rayleigh–Jeans in the UV (infinite total energy).
- Planck's single formula interpolates both limits and contains two new constants, $h$ and $k_{\rm B}$ (Eqs. 1.1.5–1.1.6) — the birth of quantum theory.
- Graphical reading: one log–log spectral map + two scaling plots (slope 4, hyperbola $1/T$) capture the whole pre-1900 contradiction.

## Gaps
- Wien's displacement "adiabatic argument": derive $\nu \propto 1/\ell$ for a slowly expanding cavity (need the Doppler / piston work) — Weinberg only quotes the result.
- Historical data: where exactly did Rubens & Kurlbaum (1900) measure relative to the Wien/Planck curves? Try digitizing the original figure.
- Show that equipartition *assumes* continuity of oscillator energies — where exactly does the discrete sum (Weinberg Eq. 1.1.7) change the high-$\nu$ behavior?